In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import pathlib as path
import plotly.express as px


In [7]:

file_path = r"C:\Users\heryn\OneDrive\Documents\Semester 1 2026\FIT 5120\iteration 1 brainboost\project brain health\Measured-physical-activity-and-sleep-all\MPASDC01.xlsx"

df = pd.read_excel(file_path, sheet_name="Table 1.1_Proportions", header=None)

print(df.head(20))

                                                   0   \
0   This tab contains table 1.1, proportion of per...   
1                     Australian Bureau of Statistics   
2   Table 1.1 Physical activity, by age and sex(a)...   
3   National Nutrition and Physical Activity Surve...   
4                                                 NaN   
5                                                 NaN   
6                                                 NaN   
7                            Average daily inactivity   
8                                   Less than 9 hours   
9                             9 to less than 10 hours   
10                           10 to less than 11 hours   
11                           11 to less than 12 hours   
12                           12 to less than 13 hours   
13                           13 to less than 14 hours   
14                           14 to less than 15 hours   
15                                   15 hours or more   
16                             

In [8]:


# use row 5 for age-group column names
columns = ["category"] + df.iloc[5, 1:].tolist()

# keep the table body only
table = df.iloc[7:55, 0:len(columns)].copy()
table.columns = columns

# clean column names
table.columns = [str(col).strip() for col in table.columns]

# clean category text
table["category"] = table["category"].astype(str).str.strip()

# keep only 18–24 column
table_18_24 = table[["category", "18–24"]].copy()
table_18_24 = table_18_24.rename(columns={"18–24": "value"})

# remove blank rows
table_18_24 = table_18_24.dropna(subset=["category"])

# remove footnotes and copyright rows
table_18_24 = table_18_24[
    ~table_18_24["category"].str.contains("Footnotes|Commonwealth of Australia", case=False, na=False)
]

# remove hash symbol from values like #1,8
table_18_24["value"] = (
    table_18_24["value"]
    .astype(str)
    .str.replace("#", "", regex=False)
    .str.replace(",", ".", regex=False)
)

# convert to numeric where possible
table_18_24["value"] = pd.to_numeric(table_18_24["value"], errors="coerce")

print(table_18_24.head(30))

                                             category  value
7                            Average daily inactivity    NaN
8                                   Less than 9 hours    7.2
9                             9 to less than 10 hours   11.0
10                           10 to less than 11 hours   17.1
11                           11 to less than 12 hours   29.7
12                           12 to less than 13 hours   11.1
13                           13 to less than 14 hours   12.5
14                           14 to less than 15 hours    7.1
15                                   15 hours or more    4.4
16                                      Total persons  100.0
17            Average daily inactivity (Mean – h:min)    NaN
18              Average daily light physical activity    NaN
19                                  Less than 2 hours   28.3
20                             2 to less than 3 hours   43.7
21                             3 to less than 4 hours   19.9
22                      

In [9]:
# -----------------------------
# Create metric groups
# -----------------------------
current_group = None
groups = []

for cat in table_18_24["category"]:
    if "Average daily inactivity" in cat and "Mean" not in cat:
        current_group = "inactivity"
        groups.append(None)
    elif "Average daily light physical activity" in cat and "Mean" not in cat:
        current_group = "light_activity"
        groups.append(None)
    elif "Average daily moderate and vigorous physical activity" in cat and "Mean" not in cat:
        current_group = "moderate_vigorous"
        groups.append(None)
    elif "Average daily moderate physical activity" in cat and "Mean" not in cat:
        current_group = "moderate"
        groups.append(None)
    elif "Average daily vigorous physical activity" in cat and "Mean" not in cat:
        current_group = "vigorous"
        groups.append(None)
    else:
        groups.append(current_group)

table_18_24["metric_group"] = groups

# -----------------------------
# Remove rows you do not need
# -----------------------------
cleaned = table_18_24[
    ~table_18_24["category"].str.contains(
        "Average daily|Total persons|Footnotes|Commonwealth of Australia",
        case=False,
        na=False
    )
].copy()

# Remove rows without metric group
cleaned = cleaned.dropna(subset=["metric_group"])

# Reorder columns
cleaned = cleaned[["metric_group", "category", "value"]]

print(cleaned)

         metric_group                                           category  \
8          inactivity                                  Less than 9 hours   
9          inactivity                            9 to less than 10 hours   
10         inactivity                           10 to less than 11 hours   
11         inactivity                           11 to less than 12 hours   
12         inactivity                           12 to less than 13 hours   
13         inactivity                           13 to less than 14 hours   
14         inactivity                           14 to less than 15 hours   
15         inactivity                                   15 hours or more   
19     light_activity                                  Less than 2 hours   
20     light_activity                             2 to less than 3 hours   
21     light_activity                             3 to less than 4 hours   
22     light_activity                                    4 hours or more   
26  moderate

In [10]:
cleaned.head(20)

,metric_group,category,value
8,inactivity,Less than 9 hours,7.2
9,inactivity,9 to less than 10 hours,11.0
10,inactivity,10 to less than 11 hours,17.1
11,inactivity,11 to less than 12 hours,29.7
12,inactivity,12 to less than 13 hours,11.1
13,inactivity,13 to less than 14 hours,12.5
14,inactivity,14 to less than 15 hours,7.1
15,inactivity,15 hours or more,4.4
19,light_activity,Less than 2 hours,28.3
20,light_activity,2 to less than 3 hours,43.7


In [11]:
cleaned.describe()

,value
count,30.000000
mean,16.673333
std,12.205791
min,1.800000
25%,9.900000
50%,13.950000
75%,19.500000
max,59.600000


In [12]:
import plotly.express as px

# filter inactivity
inactivity = cleaned[cleaned["metric_group"] == "inactivity"].copy()

fig = px.bar(
    inactivity,
    x="value",
    y="category",
    orientation="h",
    text="value",
    title="Daily Inactivity Distribution for Ages 18–24",
    labels={"value": "Percentage", "category": "Inactivity duration"},
    hover_data={"value": ":.1f"}
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    yaxis=dict(categoryorder="array", categoryarray=inactivity["category"][::-1]),
    template="plotly_white",
    height=550,
    title_x=0.5
)

fig.show()

In [13]:

mvpa = cleaned[cleaned["metric_group"] == "moderate_vigorous"].copy()

fig = px.bar(
    mvpa,
    x="value",
    y="category",
    orientation="h",
    text="value",
    title="Moderate and Vigorous Physical Activity for Ages 18–24",
    labels={"value": "Percentage", "category": "Activity duration"},
    hover_data={"value": ":.1f"}
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    yaxis=dict(categoryorder="array", categoryarray=mvpa["category"][::-1]),
    template="plotly_white",
    height=550,
    title_x=0.5
)

fig.show()

In [14]:
import plotly.express as px

main_cleaned = cleaned[
    cleaned["metric_group"].isin(["inactivity", "moderate_vigorous"])
].copy()

main_cleaned["metric_group"] = main_cleaned["metric_group"].replace({
    "inactivity": "Daily Inactivity",
    "moderate_vigorous": "Moderate + Vigorous Activity"
})

fig = px.bar(
    main_cleaned,
    x="category",
    y="value",
    color="metric_group",
    barmode="group",
    text="value",
    title="Movement Profile for Ages 18–24",
    labels={
        "category": "Duration band",
        "value": "Percentage",
        "metric_group": "Metric"
    },
    hover_data={"value": ":.1f"}
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=650,
    title_x=0.5,
    xaxis_tickangle=-30
)

fig.show()

In [15]:
import plotly.graph_objects as go

# filter inactivity only
inactivity = cleaned[cleaned["metric_group"] == "inactivity"].copy()

# average inactivity for 18–24
avg_inactivity = 11.38

fig = go.Figure()

# bar chart
fig.add_trace(go.Bar(
    x=inactivity["category"],
    y=inactivity["value"],
    text=inactivity["value"],
    textposition="outside",
    name="Percentage of 18–24",
    hovertemplate="Band: %{x}<br>Percentage: %{y:.1f}%<extra></extra>"
))

# vertical line for average
fig.add_vline(
    x=3,  # this will be placed around the 11 to <12 hours band
    line_width=3,
    line_dash="dash",
    annotation_text=f"Average inactivity = {avg_inactivity} hours",
    annotation_position="top"
)

fig.update_layout(
    title="Daily Inactivity Distribution for Ages 18–24",
    xaxis_title="Inactivity duration",
    yaxis_title="Percentage",
    template="plotly_white",
    height=600,
    title_x=0.5
)

fig.show()

In [18]:
import plotly.graph_objects as go
import pandas as pd

# Prepare data for line chart with averages
main_cleaned = cleaned[
    cleaned["metric_group"].isin(["inactivity", "moderate_vigorous"])
].copy()

main_cleaned["metric_group"] = main_cleaned["metric_group"].replace({
    "inactivity": "Daily Inactivity",
    "moderate_vigorous": "Moderate + Vigorous Activity"
})

# Calculate average for each metric group
averages = main_cleaned.groupby("metric_group")["value"].mean().reset_index()
averages.columns = ["metric_group", "average"]

print("Averages for each category:")
print(averages)

# Create line chart
fig = go.Figure()

for metric in main_cleaned["metric_group"].unique():
    metric_data = main_cleaned[main_cleaned["metric_group"] == metric]
    
    fig.add_trace(go.Scatter(
        x=metric_data["category"],
        y=metric_data["value"],
        mode="lines+markers",
        name=metric,
        line=dict(width=3),
        marker=dict(size=8),
        hovertemplate="Duration: %{x}<br>Percentage: %{y:.1f}%<extra></extra>"
    ))

# Add average lines
for _, row in averages.iterrows():
    fig.add_hline(
        y=row["average"],
        line_dash="dash",
        annotation_text=f"{row['metric_group']} Avg: {row['average']:.2f}%",
        annotation_position="right"
    )

fig.update_layout(
    title="Movement Profile for Ages 18–24",
    xaxis_title="Duration band",
    yaxis_title="Percentage",
    template="plotly_white",
    height=600,
    title_x=0.5,
    hovermode="x unified",
    xaxis_tickangle=-30
)

fig.show()

Averages for each category:
                   metric_group    average
0              Daily Inactivity  12.512500
1  Moderate + Vigorous Activity  14.285714
